# 01b · Prepare the AWE industrial dataset (6 clients, spike backdoor)
The dataset repo has this layout (adapted to in this project): `Dataset/normal` (7 files, state 0), `Dataset/1chip` (8 files),
`Dataset/2chips … 5chips` (8 files each). The official FuGuard code looked for folders `0chips…5chips`, so it could not read this layout
(and silently skipped missing files); `awe_prep.py` maps `normal → state 0`, `1chip → state 1`, … and matches names case-insensitively
(`Normal_0.json` vs `normal_1.json`).

Get the data on Colab (about 2 GB, stored on the VM): `colab.fetch_awe_dataset()` clones the GitHub repo, or pass your own copy:
`colab.fetch_awe_dataset("/content/drive/MyDrive/NGIT-2022-AWE-Industrial-Dataset-master.zip")` (a zip or a folder).

Pipeline (unchanged from the repo): z-score → 4th-order low-pass (250 Hz) → downsample 4 kHz→500 Hz → windows of 30 rows (stride 10) × 1000 features.
Client *k* holds mostly state *k* (groups 0-3) plus random other groups. Attacker `client_2`: spike trigger (+20 on several positions, noise σ=0.5)
on **all** its windows → class 5. Groups 0-6 train (state 0: 0-5) and the last file of each state is the test set.

Fixes vs the repo: config keys `SAVE_FOLDER / CLIENTS / SEED` did not exist (KeyError); trigger noise uses a seeded RNG; `num_clients` is 6.
Tested end-to-end on a *fake* tree that mirrors the real listing; the JSON field names `vibr_UB` / `vibr_LB` come from the authors' code and are not verified on the real files
(if they differ you get an error that prints the keys found - then open the dataset's `Read_Data.py`).

In [ ]:
import os, sys, pathlib, importlib.util, subprocess
_cwd = pathlib.Path.cwd().resolve()
ROOT = next((p for p in [_cwd, *_cwd.parents, pathlib.Path("/content/FuGuard_Reproduction")]
             if (p / "fuguard_repro").exists()), None)
if ROOT is None:
    raise RuntimeError("Project not found. On Colab: run COLAB_START_HERE.ipynb first (it unzips to /content/FuGuard_Reproduction).")
os.environ.setdefault("FUGUARD_ROOT", str(ROOT))
sys.path.insert(0, str(ROOT))
for _mod in ("geomloss", "diffusers"):          # a Colab runtime restart wipes pip installs -> re-install if missing
    if importlib.util.find_spec(_mod) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements-colab.txt")], check=True)
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=False)
        break
print("project root:", ROOT)

In [ ]:
from fuguard_repro import colab
colab.fetch_awe_dataset()      # or colab.fetch_awe_dataset("/path/to/zip-or-folder")

In [ ]:
from fuguard_repro.config import awe_config
from fuguard_repro.data_awe import prepare_awe_data
prepare_awe_data(awe_config())